# 34 - Stage D: does the story survive a 2x token budget?

Sharpest objection to the fixed-budget design: "at fixed tokens, growing N means growing
under-training - your saturation/compression may be an optimization artifact, not architecture."
Direct test: double the budget (12000 steps x batch 16 x block 256 = 49.2M tokens, warmup 400)
and re-measure the key points.

Predictions committed in advance (from the 1x data + Finding F4):
1. conv's plateau persists (Delta ~ -10% at 30M, still negative at 91M);
2. diff still loses at every size here (its curve shifts but sign stays positive at <=91M);
3. the conv crossover moves LEFT (at 10M, conv's Delta should be closer to zero or negative
   than the +5.1% measured at 1x) - budget helps conv express its benefit earlier.


In [1]:
%matplotlib inline
import os, math, time, csv, json, gc
from contextlib import nullcontext
import numpy as np
import torch
from model import Config, GPT, count_params

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
D = "../dataset"
train_data = torch.from_numpy(np.load(D + "/wt103_bpe_train.npy"))
val_data = torch.from_numpy(np.load(D + "/wt103_bpe_val.npy"))
vocab = json.load(open(D + "/wt103_bpe_meta.json"))["vocab"]
print(f"vocab={vocab}  train={len(train_data):,}  val={len(val_data):,}")

BLOCK, BATCH = 256, 16
CKPT_DIR = "../results/budget2x_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

COMBO = {"pos_encoding": "rope", "activation": "swiglu", "norm_placement": "pre", "norm_type": "rmsnorm"}
VARIANTS = {"combo": COMBO, "combo_conv3": {**COMBO, "conv_kernel": 3}, "combo_diff": {**COMBO, "attention": "diff"}}
SIZES = {"10M": (320, 6, 8), "30M": (512, 8, 8), "90M": (768, 12, 12)}
RUNS_CSV = "../results/budget2x_runs.csv"
if not os.path.exists(RUNS_CSV):
    with open(RUNS_CSV, "w", newline="") as f:
        csv.writer(f).writerow(["size", "variant", "seed", "params_M", "val_ppl"])

def get_batch(split, gen):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK, (BATCH,), generator=gen)
    x = torch.stack([d[i:i + BLOCK] for i in ix]); y = torch.stack([d[i + 1:i + 1 + BLOCK] for i in ix])
    return x.to(device), y.to(device)
def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

def train_one(name, size_cfg, over, seed=0, max_iters=12000, eval_iters=200,
              lr=3e-4, warmup=400, min_lr=3e-5, ckpt_every=500):
    ne, nl, nh = size_cfg
    ckpt_path = f"{CKPT_DIR}/{name}_s{seed}.pt"
    torch.manual_seed(seed)
    cfg = Config(vocab_size=vocab, block_size=BLOCK, n_layer=nl, n_head=nh, n_embd=ne,
                 dropout=0.1, **over)
    model = GPT(cfg).to(device)
    pm = count_params(model) / 1e6
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
    tg = torch.Generator().manual_seed(7777 + seed)
    start_it = 0
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location="cpu")   # cpu: RNG state must stay a CPU ByteTensor
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        tg.set_state(ck["tg"].cpu()); start_it = ck["iter"]
        print(f"    [{name} s{seed}] RESUMED from {start_it}", flush=True)
    def get_lr(it):
        if it < warmup: return lr * (it + 1) / warmup
        r = (it - warmup) / max(1, max_iters - warmup)
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * r))
    def save_ckpt(it):
        tmp = ckpt_path + ".tmp"
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "iter": it, "tg": tg.get_state()}, tmp)
        os.replace(tmp, ckpt_path)
    @torch.no_grad()
    def ev():
        model.eval(); g = torch.Generator().manual_seed(4242); tot = 0.0
        for _ in range(eval_iters):
            x, y = get_batch("val", g)
            with autocast_ctx():
                _, loss = model(x, y)
            tot += loss.item()
        model.train(); return tot / eval_iters
    for it in range(start_it, max_iters):
        for grp in opt.param_groups: grp["lr"] = get_lr(it)
        x, y = get_batch("train", tg)
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if (it + 1) % ckpt_every == 0: save_ckpt(it + 1)
    ppl = math.exp(ev())
    if os.path.exists(ckpt_path): os.remove(ckpt_path)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return ppl, pm

def run2x(sname, vname, seed):
    done = set()
    with open(RUNS_CSV) as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if len(row) == 5: done.add((row[0], row[1], row[2]))
    if (sname, vname, str(seed)) in done:
        print(f"  {sname} {vname} s{seed}: done, skip", flush=True); return
    t0 = time.time()
    p, pm = train_one(f"2x_{sname}_{vname}", SIZES[sname], VARIANTS[vname], seed=seed)
    with open(RUNS_CSV, "a", newline="") as f:
        csv.writer(f).writerow([sname, vname, seed, f"{pm:.3f}", f"{p:.4f}"])
    print(f"  {sname:4s} {vname:12s} seed {seed}: ppl {p:.3f} ({pm:.1f}M) | {(time.time()-t0)/60:.1f} min", flush=True)
print("ready")


device: cuda
vocab=8192  train=14,923,628  val=281,169
ready


In [2]:
run2x("10M", "combo", 0)

<repo>\.venv\Lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


  10M  combo        seed 0: ppl 48.246 (10.0M) | 10.1 min


In [3]:
run2x("10M", "combo", 1)

  10M  combo        seed 1: ppl 48.084 (10.0M) | 10.2 min


In [4]:
run2x("10M", "combo_conv3", 0)

  10M  combo_conv3  seed 0: ppl 48.329 (10.0M) | 11.6 min


In [5]:
run2x("10M", "combo_conv3", 1)

  10M  combo_conv3  seed 1: ppl 48.402 (10.0M) | 11.5 min


In [6]:
run2x("10M", "combo_diff", 0)

  10M  combo_diff   seed 0: ppl 51.537 (10.0M) | 15.1 min


In [7]:
run2x("10M", "combo_diff", 1)

  10M  combo_diff   seed 1: ppl 53.466 (10.0M) | 15.1 min


In [8]:
run2x("30M", "combo", 0)

  30M  combo        seed 0: ppl 39.200 (29.4M) | 18.3 min


In [9]:
run2x("30M", "combo", 1)

  30M  combo        seed 1: ppl 38.718 (29.4M) | 18.3 min


In [10]:
run2x("30M", "combo_conv3", 0)

  30M  combo_conv3  seed 0: ppl 37.775 (29.4M) | 20.5 min


In [11]:
run2x("30M", "combo_conv3", 1)

  30M  combo_conv3  seed 1: ppl 37.860 (29.4M) | 20.5 min


In [12]:
run2x("30M", "combo_diff", 0)

  30M  combo_diff   seed 0: ppl 39.784 (29.4M) | 25.5 min


In [13]:
run2x("30M", "combo_diff", 1)

  30M  combo_diff   seed 1: ppl 39.670 (29.4M) | 25.4 min


In [14]:
run2x("90M", "combo", 0)

  90M  combo        seed 0: ppl 34.210 (91.3M) | 43.3 min


In [15]:
run2x("90M", "combo_conv3", 0)

  90M  combo_conv3  seed 0: ppl 34.032 (91.4M) | 48.7 min


In [16]:
run2x("90M", "combo_diff", 0)

  90M  combo_diff   seed 0: ppl 34.835 (91.3M) | 61.9 min


In [17]:
# === Analysis: 1x vs 2x budget ===
def load_kv(fn, keyfn, valfn):
    out = {}
    with open(fn) as f:
        r = csv.reader(f); next(r, None)
        for row in r: out[keyfn(row)] = valfn(row)
    return out
b2 = load_kv(RUNS_CSV, lambda r: (r[0], r[1], r[2]), lambda r: float(r[4]))
lad = load_kv("../results/ladder6000_runs.csv", lambda r: (r[0], r[1], r[2]), lambda r: float(r[4]))
sc0 = load_kv("../results/scale_confirm_runs.csv", lambda r: r[0], lambda r: float(r[2]))
sc1 = load_kv("../results/scale_confirm_s1_runs.csv", lambda r: r[0], lambda r: float(r[2]))
v90 = load_kv("../results/val90_runs.csv", lambda r: (r[0], r[1]), lambda r: float(r[3]))

def d1x(sz, variant):
    if sz == "10M":
        return np.mean([100*(lad[(sz,variant,s)]-lad[(sz,"combo",s)])/lad[(sz,"combo",s)] for s in ("0","1")])
    if sz == "30M":
        return np.mean([100*((sc0 if s=="0" else sc1)[variant]-(sc0 if s=="0" else sc1)["combo"])/(sc0 if s=="0" else sc1)["combo"] for s in ("0","1")])
    return np.mean([100*(v90[(variant,s)]-v90[("combo",s)])/v90[("combo",s)] for s in ("0","1")])
def d2x(sz, variant):
    ds = [100*(b2[(sz,variant,s)]-b2[(sz,"combo",s)])/b2[(sz,"combo",s)]
          for s in ("0","1") if (sz,variant,s) in b2 and (sz,"combo",s) in b2]
    return (float(np.mean(ds)), len(ds)) if ds else (None, 0)

print("=== effect sizes: 1x budget (24.6M tok) vs 2x budget (49.2M tok) ===")
print(f"{'':14s} {'size':5s} {'1x':>8s} {'2x':>8s}")
checks = []
for variant, label in [("combo_conv3", "conv"), ("combo_diff", "diff")]:
    for sz in ["10M", "30M", "90M"]:
        v2, n = d2x(sz, variant)
        if v2 is None: continue
        v1 = d1x(sz, variant)
        print(f"  {label:12s} {sz:5s} {v1:+8.2f} {v2:+8.2f}   ({n} seeds)")
        checks.append((variant, sz, v1, v2))
print()
conv30 = next((c for c in checks if c[0]=="combo_conv3" and c[1]=="30M"), None)
diffs = [c for c in checks if c[0]=="combo_diff"]
conv10 = next((c for c in checks if c[0]=="combo_conv3" and c[1]=="10M"), None)
if conv30:
    ok = conv30[3] < -5
    print(f"P1 conv plateau persists at 2x: 30M Delta {conv30[3]:+.2f}% -> " + ("HOLDS" if ok else "WEAKENED - report honestly"))
if diffs:
    ok = all(c[3] > 0 for c in diffs)
    print(f"P2 diff still loses everywhere at 2x: " + ("HOLDS" if ok else "VIOLATED at some size - report honestly"))
if conv10:
    ok = conv10[3] < conv10[2]
    print(f"P3 crossover moves left: conv@10M {conv10[2]:+.2f}% (1x) -> {conv10[3]:+.2f}% (2x): " + ("HOLDS" if ok else "VIOLATED"))


=== effect sizes: 1x budget (24.6M tok) vs 2x budget (49.2M tok) ===
               size        1x       2x
  conv         10M      +5.13    +0.42   (2 seeds)
  conv         30M     -10.23    -2.93   (2 seeds)
  conv         90M      -9.85    -0.52   (1 seeds)
  diff         10M     +21.82    +9.01   (2 seeds)
  diff         30M      +3.95    +1.98   (2 seeds)
  diff         90M      +1.48    +1.83   (1 seeds)

P1 conv plateau persists at 2x: 30M Delta -2.93% -> WEAKENED - report honestly
P2 diff still loses everywhere at 2x: HOLDS
P3 crossover moves left: conv@10M +5.13% (1x) -> +0.42% (2x): HOLDS


## Reading this

If P1-P3 hold, the saturation/crossover story is a property of the (architecture, size, budget)
surface, not an under-training artifact - and F4 becomes a quantified, two-budget result. Any
violation is reported as-is; the analysis prints the honest verdict either way.
